# Model Validation: Batch SMC Results

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/anglersim=/path/to/anglersim"` at the top of any cell.

This notebook analyses the output of `cmd/batchsmc` — posterior parameter estimates across all fitted sites.
We check:
1. Summary statistics and fit quality across sites
2. Parameter distributions — are posteriors sensible or railing against priors?
3. Site quality flags — low marginal likelihood, extreme parameters, high uncertainty
4. Posterior predictive checks — does the fitted model reproduce observed dynamics?

## 1. Load batch results and summarise

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"math"
	"os"
	"sort"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"gonum.org/v1/gonum/stat"
)

%%

f, err := os.Open("../dat/batch_smc_results.csv")
if err != nil {
	panic(err)
}
defer f.Close()

r := csv.NewReader(f)
records, _ := r.ReadAll()

// Count statuses
statusCounts := make(map[string]int)
for _, rec := range records {
	statusCounts[rec[3]]++
}
fmt.Printf("Total sites: %d\n", len(records))
for s, c := range statusCounts {
	fmt.Printf("  %s: %d\n", s, c)
}

// Collect OK sites' logZ and parameter summaries
paramNames := []string{"growth_rate", "density_dependence", "beta_flow", "beta_temp", "beta_do", "process_noise_sd", "obs_noise_var"}
nParams := len(paramNames)

var logZs []float64
var numYears []float64
means := make([][]float64, nParams)
stds := make([][]float64, nParams)
for i := range nParams {
	means[i] = make([]float64, 0)
	stds[i] = make([]float64, 0)
}

for _, rec := range records {
	if rec[3] != "OK" {
		continue
	}
	lz, _ := strconv.ParseFloat(rec[2], 64)
	logZs = append(logZs, lz)
	ny, _ := strconv.ParseFloat(rec[1], 64)
	numYears = append(numYears, ny)
	for i := range nParams {
		m, _ := strconv.ParseFloat(rec[4+2*i], 64)
		s, _ := strconv.ParseFloat(rec[5+2*i], 64)
		means[i] = append(means[i], m)
		stds[i] = append(stds[i], s)
	}
}

fmt.Printf("\nOK sites: %d\n\n", len(logZs))

// Log marginal likelihood summary
sort.Float64s(logZs)
fmt.Printf("Log marginal likelihood (logZ):\n")
fmt.Printf("  min=%.2f  Q25=%.2f  median=%.2f  Q75=%.2f  max=%.2f\n",
	logZs[0],
	logZs[len(logZs)/4],
	logZs[len(logZs)/2],
	logZs[3*len(logZs)/4],
	logZs[len(logZs)-1],
)
fmt.Printf("  mean=%.2f  std=%.2f\n", stat.Mean(logZs, nil), math.Sqrt(stat.Variance(logZs, nil)))

// Parameter summary table
fmt.Printf("\nPosterior mean summary across sites:\n")
fmt.Printf("%-22s %8s %8s %8s %8s %8s\n", "Parameter", "Min", "Q25", "Median", "Q75", "Max")
for i, name := range paramNames {
	sorted := make([]float64, len(means[i]))
	copy(sorted, means[i])
	sort.Float64s(sorted)
	n := len(sorted)
	fmt.Printf("%-22s %8.4f %8.4f %8.4f %8.4f %8.4f\n",
		name, sorted[0], sorted[n/4], sorted[n/2], sorted[3*n/4], sorted[n-1])
}

// Posterior std summary
fmt.Printf("\nPosterior std summary across sites:\n")
fmt.Printf("%-22s %8s %8s %8s %8s %8s\n", "Parameter", "Min", "Q25", "Median", "Q75", "Max")
for i, name := range paramNames {
	sorted := make([]float64, len(stds[i]))
	copy(sorted, stds[i])
	sort.Float64s(sorted)
	n := len(sorted)
	fmt.Printf("%-22s %8.4f %8.4f %8.4f %8.4f %8.4f\n",
		name, sorted[0], sorted[n/4], sorted[n/2], sorted[3*n/4], sorted[n-1])
}

## 2. Parameter distributions across sites

In [ ]:
import (
	"encoding/csv"
	"os"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Load batch results
f, _ := os.Open("../dat/batch_smc_results.csv")
defer f.Close()
r := csv.NewReader(f)
records, _ := r.ReadAll()

paramNames := []string{"growth_rate", "density_dependence", "beta_flow", "beta_temp", "beta_do", "process_noise_sd", "obs_noise_var"}

// Build a long-form dataframe: SITE_INDEX, PARAM_MEAN, PARAM_NAME
var siteIndices []float64
var paramMeans []float64
var paramLabels []string

idx := 0
for _, rec := range records {
	if rec[3] != "OK" {
		continue
	}
	for i, name := range paramNames {
		m, _ := strconv.ParseFloat(rec[4+2*i], 64)
		siteIndices = append(siteIndices, float64(idx))
		paramMeans = append(paramMeans, m)
		paramLabels = append(paramLabels, name)
	}
	idx++
}

// One scatter plot per parameter: site index vs posterior mean
for _, name := range paramNames {
	var xs, ys []float64
	for j := range siteIndices {
		if paramLabels[j] == name {
			xs = append(xs, siteIndices[j])
			ys = append(ys, paramMeans[j])
		}
	}
	df := dataframe.New(
		series.New(xs, series.Float, "SITE_INDEX"),
		series.New(ys, series.Float, "POSTERIOR_MEAN"),
	)
	scatter := analysis.NewScatterPlotFromDataFrame(&df, "SITE_INDEX", "POSTERIOR_MEAN")
	scatter.SetGlobalOptions(
		charts.WithTitleOpts(opts.Title{
			Title: "Posterior mean: " + name,
			Bottom: "1%",
		}),
		charts.WithYAxisOpts(opts.YAxis{
			Min: floats.Min(ys),
			Max: floats.Max(ys),
		}),
	)
	gonb_echarts.Display(scatter, "width: 1024px; height:300px; background: white;")
}

## 3. LogZ vs number of years — does fit quality depend on data length?

In [ ]:
import (
	"encoding/csv"
	"os"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

f, _ := os.Open("../dat/batch_smc_results.csv")
defer f.Close()
r := csv.NewReader(f)
r.Read() // skip header
records, _ := r.ReadAll()

var years, logZs []float64
for _, rec := range records {
	if rec[3] != "OK" {
		continue
	}
	ny, _ := strconv.ParseFloat(rec[1], 64)
	lz, _ := strconv.ParseFloat(rec[2], 64)
	years = append(years, ny)
	logZs = append(logZs, lz)
}

df := dataframe.New(
	series.New(years, series.Float, "NUM_YEARS"),
	series.New(logZs, series.Float, "LOG_MARGINAL_LIK"),
)

scatter := analysis.NewScatterPlotFromDataFrame(&df, "NUM_YEARS", "LOG_MARGINAL_LIK")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Log Marginal Likelihood vs Number of Years",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: floats.Min(logZs),
		Max: floats.Max(logZs),
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(years) - 1,
		Max: floats.Max(years) + 1,
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

## 4. Site quality flags

Flag sites with potential issues:
- Very low logZ (bottom 10%) — poor model fit
- Extreme parameter values (near prior boundaries)
- Very high posterior std (uninformative data)

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"os"
	"sort"
	"strconv"
)

func abs(x float64) float64 {
	if x < 0 {
		return -x
	}
	return x
}

%%

f, _ := os.Open("../dat/batch_smc_results.csv")
defer f.Close()
r := csv.NewReader(f)
r.Read()
records, _ := r.ReadAll()

paramNames := []string{"growth_rate", "density_dependence", "beta_flow", "beta_temp", "beta_do", "process_noise_sd", "obs_noise_var"}

// Prior bounds for detecting railing (from CLAUDE.md parameter table)
// [low_warning, high_warning] — values near prior edges
priorWarnings := map[string][2]float64{
	"growth_rate":         {-1.8, 4.5},
	"density_dependence":  {0.01, 50.0},
	"beta_flow":           {-1.8, 1.8},
	"beta_temp":           {-1.8, 1.8},
	"beta_do":             {-1.8, 1.8},
	"process_noise_sd":    {0.001, 2.0},
	"obs_noise_var":       {0.001, 10.0},
}

type siteInfo struct {
	SiteID   string
	NumYears string
	LogZ     float64
	Means    [7]float64
	Stds     [7]float64
	Flags    []string
}

// Collect logZs for percentile calculation
var allLogZ []float64
for _, rec := range records {
	if rec[3] != "OK" {
		continue
	}
	lz, _ := strconv.ParseFloat(rec[2], 64)
	allLogZ = append(allLogZ, lz)
}
sortedLogZ := make([]float64, len(allLogZ))
copy(sortedLogZ, allLogZ)
sort.Float64s(sortedLogZ)
logZThresh10 := sortedLogZ[len(sortedLogZ)/10] // bottom 10%

// Flag sites
var flagged []siteInfo
for _, rec := range records {
	if rec[3] != "OK" {
		continue
	}
	si := siteInfo{SiteID: rec[0], NumYears: rec[1]}
	si.LogZ, _ = strconv.ParseFloat(rec[2], 64)
	for i := range 7 {
		si.Means[i], _ = strconv.ParseFloat(rec[4+2*i], 64)
		si.Stds[i], _ = strconv.ParseFloat(rec[5+2*i], 64)
	}

	// Flag: low logZ
	if si.LogZ < logZThresh10 {
		si.Flags = append(si.Flags, fmt.Sprintf("low_logZ(%.2f)", si.LogZ))
	}

	// Flag: parameters near prior boundaries
	for i, name := range paramNames {
		bounds := priorWarnings[name]
		if si.Means[i] < bounds[0] {
			si.Flags = append(si.Flags, fmt.Sprintf("%s_low(%.4f)", name, si.Means[i]))
		}
		if si.Means[i] > bounds[1] {
			si.Flags = append(si.Flags, fmt.Sprintf("%s_high(%.4f)", name, si.Means[i]))
		}
	}

	// Flag: very high posterior std relative to mean (coefficient of variation > 1)
	for i, name := range paramNames {
		if si.Means[i] != 0 && si.Stds[i]/abs(si.Means[i]) > 1.0 {
			si.Flags = append(si.Flags, fmt.Sprintf("%s_wide_posterior", name))
		}
	}

	if len(si.Flags) > 0 {
		flagged = append(flagged, si)
	}
}

fmt.Printf("Flagged sites: %d / %d\n\n", len(flagged), len(allLogZ))
fmt.Printf("LogZ bottom-10%% threshold: %.2f\n\n", logZThresh10)

// Show up to 30 most-flagged sites
sort.Slice(flagged, func(i, j int) bool {
	return len(flagged[i].Flags) > len(flagged[j].Flags)
})
n := len(flagged)
if n > 30 {
	n = 30
}
fmt.Printf("Top %d most-flagged sites:\n", n)
fmt.Printf("%-10s %-8s %10s  %s\n", "SITE_ID", "YEARS", "logZ", "FLAGS")
for _, si := range flagged[:n] {
	fmt.Printf("%-10s %-8s %10.2f  %v\n", si.SiteID, si.NumYears, si.LogZ, si.Flags)
}

// Summarise flag types
flagCounts := make(map[string]int)
for _, si := range flagged {
	for _, fl := range si.Flags {
		// Normalise to category
		cat := fl
		for _, p := range paramNames {
			if len(fl) > len(p) && fl[:len(p)] == p {
				if fl[len(p):len(p)+4] == "_low" {
					cat = p + "_low"
				} else if fl[len(p):len(p)+5] == "_high" {
					cat = p + "_high"
				} else if fl[len(p):len(p)+5] == "_wide" {
					cat = p + "_wide_posterior"
				}
			}
		}
		if cat[:3] == "low" {
			cat = "low_logZ"
		}
		flagCounts[cat]++
	}
}
fmt.Printf("\nFlag summary:\n")
for cat, c := range flagCounts {
	fmt.Printf("  %-35s %d sites\n", cat, c)
}

## 5. Posterior predictive check

For a selection of sites, simulate forward from the posterior mean parameters and compare to observed log-density time series. This is a basic sanity check — the model should roughly track the data.

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"math"
	"os"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Load batch results
bf, _ := os.Open("../dat/batch_smc_results.csv")
defer bf.Close()
br := csv.NewReader(bf)
br.Read()
batchRecords, _ := br.ReadAll()

// Index results by site ID
type fitResult struct {
	SiteID   int
	LogZ     float64
	Means    [7]float64
}
fits := make(map[int]fitResult)
for _, rec := range batchRecords {
	if rec[3] != "OK" {
		continue
	}
	id, _ := strconv.Atoi(rec[0])
	fr := fitResult{SiteID: id}
	fr.LogZ, _ = strconv.ParseFloat(rec[2], 64)
	for i := range 7 {
		fr.Means[i], _ = strconv.ParseFloat(rec[4+2*i], 64)
	}
	fits[id] = fr
}

// Load all site time series
allSites := data.LoadAllSiteTimeSeries("../dat/brown_trout_panel_with_covariates.csv")

// Pick a few sites: best logZ, worst logZ, and median logZ
type idLogZ struct {
	ID   int
	LogZ float64
}
var sorted []idLogZ
for id, fr := range fits {
	sorted = append(sorted, idLogZ{id, fr.LogZ})
}
sort.Slice(sorted, func(i, j int) bool { return sorted[i].LogZ > sorted[j].LogZ })

// Select: best, 25th percentile, median, 75th percentile, worst
picks := []struct{ label string; idx int }{
	{"Best logZ", 0},
	{"Q75 logZ", len(sorted) / 4},
	{"Median logZ", len(sorted) / 2},
	{"Q25 logZ", 3 * len(sorted) / 4},
	{"Worst logZ", len(sorted) - 1},
}

for _, pick := range picks {
	siteID := sorted[pick.idx].ID
	sd := allSites[siteID]
	fr := fits[siteID]
	if sd == nil || len(sd.Years) < 2 {
		continue
	}

	// Deterministic Ricker forward simulation from posterior mean
	r0 := fr.Means[0]
	alpha := fr.Means[1]
	betas := fr.Means[2:5]
	T := len(sd.Years)

	predLogDens := make([]float64, T)
	predLogDens[0] = sd.LogDensity[0][0] // start from observed
	for t := 1; t < T; t++ {
		logN := predLogDens[t-1]
		envEffect := 0.0
		for k := 0; k < 3 && k < len(sd.Covariates[t]); k++ {
			envEffect += betas[k] * sd.Covariates[t][k]
		}
		predLogDens[t] = logN + r0 + envEffect - alpha*math.Exp(logN)
	}

	// Build dataframe with observed and predicted
	var allYears, allValues []float64
	var allLabels []string
	for t := 0; t < T; t++ {
		allYears = append(allYears, sd.Years[t], sd.Years[t])
		allValues = append(allValues, sd.LogDensity[t][0], predLogDens[t])
		allLabels = append(allLabels, "Observed", "Predicted")
	}

	df := dataframe.New(
		series.New(allYears, series.Float, "YEAR"),
		series.New(allValues, series.Float, "LOG_DENSITY"),
		series.New(allLabels, series.String, "TYPE"),
	)

	scatter := analysis.NewScatterPlotFromDataFrame(&df, "YEAR", "LOG_DENSITY", "TYPE")
	scatter.SetGlobalOptions(
		charts.WithTitleOpts(opts.Title{
			Title: fmt.Sprintf("%s — Site %d (logZ=%.2f, T=%d)", pick.label, siteID, fr.LogZ, T),
			Bottom: "1%",
		}),
		charts.WithYAxisOpts(opts.YAxis{
			Min: floats.Min(allValues) - 0.5,
			Max: floats.Max(allValues) + 0.5,
		}),
		charts.WithXAxisOpts(opts.XAxis{
			Min: floats.Min(allYears) - 1,
			Max: floats.Max(allYears) + 1,
		}),
	)
	gonb_echarts.Display(scatter, "width: 1024px; height:350px; background: white;")
}

## 6. One-step-ahead prediction errors

For each fitted site, compute the one-step-ahead prediction error: given the observed state at time t, predict t+1 using the fitted parameters, and compare to the actual observation. This gives a distribution of residuals across all sites and timesteps.

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"math"
	"os"
	"sort"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"gonum.org/v1/gonum/stat"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Load batch results
bf, _ := os.Open("../dat/batch_smc_results.csv")
defer bf.Close()
br := csv.NewReader(bf)
br.Read()
batchRecords, _ := br.ReadAll()

type fitResult struct {
	Means [7]float64
}
fits := make(map[int]fitResult)
for _, rec := range batchRecords {
	if rec[3] != "OK" {
		continue
	}
	id, _ := strconv.Atoi(rec[0])
	fr := fitResult{}
	for i := range 7 {
		fr.Means[i], _ = strconv.ParseFloat(rec[4+2*i], 64)
	}
	fits[id] = fr
}

allSites := data.LoadAllSiteTimeSeries("../dat/brown_trout_panel_with_covariates.csv")

// Compute one-step-ahead residuals for all sites
var allResiduals []float64
var siteRMSEs []float64

for id, fr := range fits {
	sd := allSites[id]
	if sd == nil || len(sd.Years) < 2 {
		continue
	}

	r0 := fr.Means[0]
	alpha := fr.Means[1]
	betas := fr.Means[2:5]

	var siteResiduals []float64
	for t := 1; t < len(sd.Years); t++ {
		logN := sd.LogDensity[t-1][0]
		envEffect := 0.0
		for k := 0; k < 3 && k < len(sd.Covariates[t]); k++ {
			envEffect += betas[k] * sd.Covariates[t][k]
		}
		pred := logN + r0 + envEffect - alpha*math.Exp(logN)
		residual := sd.LogDensity[t][0] - pred
		allResiduals = append(allResiduals, residual)
		siteResiduals = append(siteResiduals, residual)
	}

	if len(siteResiduals) > 0 {
		ss := 0.0
		for _, r := range siteResiduals {
			ss += r * r
		}
		siteRMSEs = append(siteRMSEs, math.Sqrt(ss/float64(len(siteResiduals))))
	}
}

// Summary stats for residuals
fmt.Printf("One-step-ahead residuals (all sites, all timesteps):\n")
fmt.Printf("  N = %d\n", len(allResiduals))
fmt.Printf("  Mean = %.4f\n", stat.Mean(allResiduals, nil))
fmt.Printf("  Std  = %.4f\n", math.Sqrt(stat.Variance(allResiduals, nil)))
sorted := make([]float64, len(allResiduals))
copy(sorted, allResiduals)
sort.Float64s(sorted)
n := len(sorted)
fmt.Printf("  Min=%.4f  Q25=%.4f  Median=%.4f  Q75=%.4f  Max=%.4f\n",
	sorted[0], sorted[n/4], sorted[n/2], sorted[3*n/4], sorted[n-1])

// Site-level RMSE summary
sort.Float64s(siteRMSEs)
nr := len(siteRMSEs)
fmt.Printf("\nSite-level RMSE:\n")
fmt.Printf("  N = %d sites\n", nr)
fmt.Printf("  Min=%.4f  Q25=%.4f  Median=%.4f  Q75=%.4f  Max=%.4f\n",
	siteRMSEs[0], siteRMSEs[nr/4], siteRMSEs[nr/2], siteRMSEs[3*nr/4], siteRMSEs[nr-1])

// Plot residual distribution (binned as scatter — approx histogram)
nBins := 50
rMin := sorted[0]
rMax := sorted[n-1]
binWidth := (rMax - rMin) / float64(nBins)
binCenters := make([]float64, nBins)
binCounts := make([]float64, nBins)
for _, v := range allResiduals {
	bi := int((v - rMin) / binWidth)
	if bi >= nBins {
		bi = nBins - 1
	}
	if bi < 0 {
		bi = 0
	}
	binCounts[bi]++
}
for i := range nBins {
	binCenters[i] = rMin + (float64(i)+0.5)*binWidth
}

df := dataframe.New(
	series.New(binCenters, series.Float, "RESIDUAL"),
	series.New(binCounts, series.Float, "COUNT"),
)
scatter := analysis.NewScatterPlotFromDataFrame(&df, "RESIDUAL", "COUNT")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "One-Step-Ahead Residual Distribution (all sites)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: 0.0,
		Max: floats.Max(binCounts) * 1.1,
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

// Plot site RMSE distribution
nBins2 := 30
rMin2 := siteRMSEs[0]
rMax2 := siteRMSEs[nr-1]
binWidth2 := (rMax2 - rMin2) / float64(nBins2)
bc2 := make([]float64, nBins2)
bct2 := make([]float64, nBins2)
for _, v := range siteRMSEs {
	bi := int((v - rMin2) / binWidth2)
	if bi >= nBins2 {
		bi = nBins2 - 1
	}
	if bi < 0 {
		bi = 0
	}
	bct2[bi]++
}
for i := range nBins2 {
	bc2[i] = rMin2 + (float64(i)+0.5)*binWidth2
}

df2 := dataframe.New(
	series.New(bc2, series.Float, "RMSE"),
	series.New(bct2, series.Float, "COUNT"),
)
scatter2 := analysis.NewScatterPlotFromDataFrame(&df2, "RMSE", "COUNT")
scatter2.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Site-Level RMSE Distribution",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: 0.0,
		Max: floats.Max(bct2) * 1.1,
	}),
)
gonb_echarts.Display(scatter2, "width: 1024px; height:400px; background: white;")

## 7. Held-out prediction validation

Run `go run ./cmd/validate` first to produce `dat/validation_results.csv` and `dat/validation_predictions.csv`.

This re-fits each site on a truncated time series (dropping the last K years), then forward-simulates from the posterior to predict the held-out years. Metrics:
- **RMSE / MAE** — prediction accuracy in log-density space
- **Coverage 90 / 50** — fraction of held-out observations within 90% / 50% prediction intervals

Well-calibrated prediction intervals should have coverage close to their nominal levels.

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"math"
	"os"
	"sort"
	"strconv"

	"gonum.org/v1/gonum/stat"
)

%%

// Load validation results
f, err := os.Open("../dat/validation_results.csv")
if err != nil {
	panic("Run 'go run ./cmd/validate' first: " + err.Error())
}
defer f.Close()

r := csv.NewReader(f)
r.Read() // skip header
records, _ := r.ReadAll()

// Count statuses
statusCounts := make(map[string]int)
for _, rec := range records {
	statusCounts[rec[8]]++
}
fmt.Printf("Total sites: %d\n", len(records))
for s, c := range statusCounts {
	fmt.Printf("  %s: %d\n", s, c)
}

// Collect OK sites
var rmses, maes, cov90s, cov50s, trainLogZs []float64
for _, rec := range records {
	if rec[8] != "OK" {
		continue
	}
	tlz, _ := strconv.ParseFloat(rec[3], 64)
	rmse, _ := strconv.ParseFloat(rec[4], 64)
	mae, _ := strconv.ParseFloat(rec[5], 64)
	c90, _ := strconv.ParseFloat(rec[6], 64)
	c50, _ := strconv.ParseFloat(rec[7], 64)
	trainLogZs = append(trainLogZs, tlz)
	rmses = append(rmses, rmse)
	maes = append(maes, mae)
	cov90s = append(cov90s, c90)
	cov50s = append(cov50s, c50)
}

fmt.Printf("\nOK sites: %d\n\n", len(rmses))

// RMSE summary
sort.Float64s(rmses)
n := len(rmses)
fmt.Printf("Held-out RMSE (log-density):\n")
fmt.Printf("  Min=%.4f  Q25=%.4f  Median=%.4f  Q75=%.4f  Max=%.4f\n",
	rmses[0], rmses[n/4], rmses[n/2], rmses[3*n/4], rmses[n-1])
fmt.Printf("  Mean=%.4f  Std=%.4f\n", stat.Mean(rmses, nil), math.Sqrt(stat.Variance(rmses, nil)))

// MAE summary
sort.Float64s(maes)
fmt.Printf("\nHeld-out MAE (log-density):\n")
fmt.Printf("  Min=%.4f  Q25=%.4f  Median=%.4f  Q75=%.4f  Max=%.4f\n",
	maes[0], maes[n/4], maes[n/2], maes[3*n/4], maes[n-1])

// Coverage summary
fmt.Printf("\n90%% prediction interval coverage:\n")
fmt.Printf("  Mean=%.4f  (nominal=0.90)\n", stat.Mean(cov90s, nil))
fmt.Printf("50%% prediction interval coverage:\n")
fmt.Printf("  Mean=%.4f  (nominal=0.50)\n", stat.Mean(cov50s, nil))

In [ ]:
import (
	"encoding/csv"
	"os"
	"sort"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

f, _ := os.Open("../dat/validation_results.csv")
defer f.Close()
r := csv.NewReader(f)
r.Read()
records, _ := r.ReadAll()

var rmses, cov90s []float64
for _, rec := range records {
	if rec[8] != "OK" {
		continue
	}
	rmse, _ := strconv.ParseFloat(rec[4], 64)
	c90, _ := strconv.ParseFloat(rec[6], 64)
	rmses = append(rmses, rmse)
	cov90s = append(cov90s, c90)
}

// RMSE distribution
sort.Float64s(rmses)
nBins := 30
rMin, rMax := rmses[0], rmses[len(rmses)-1]
binW := (rMax - rMin) / float64(nBins)
bc := make([]float64, nBins)
bct := make([]float64, nBins)
for _, v := range rmses {
	bi := int((v - rMin) / binW)
	if bi >= nBins { bi = nBins - 1 }
	if bi < 0 { bi = 0 }
	bct[bi]++
}
for i := range nBins {
	bc[i] = rMin + (float64(i)+0.5)*binW
}

df := dataframe.New(
	series.New(bc, series.Float, "RMSE"),
	series.New(bct, series.Float, "COUNT"),
)
scatter := analysis.NewScatterPlotFromDataFrame(&df, "RMSE", "COUNT")
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Held-Out RMSE Distribution",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Min: 0.0, Max: floats.Max(bct) * 1.1}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

// Coverage 90 distribution
sort.Float64s(cov90s)
nBins2 := 10
binW2 := 1.0 / float64(nBins2)
bc2 := make([]float64, nBins2)
bct2 := make([]float64, nBins2)
for _, v := range cov90s {
	bi := int(v / binW2)
	if bi >= nBins2 { bi = nBins2 - 1 }
	if bi < 0 { bi = 0 }
	bct2[bi]++
}
for i := range nBins2 {
	bc2[i] = (float64(i) + 0.5) * binW2
}

df2 := dataframe.New(
	series.New(bc2, series.Float, "COVERAGE_90"),
	series.New(bct2, series.Float, "COUNT"),
)
scatter2 := analysis.NewScatterPlotFromDataFrame(&df2, "COVERAGE_90", "COUNT")
scatter2.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "90% Prediction Interval Coverage Distribution (nominal=0.90)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{Min: 0.0, Max: floats.Max(bct2) * 1.1}),
	charts.WithXAxisOpts(opts.XAxis{Min: 0.0, Max: 1.0}),
)
gonb_echarts.Display(scatter2, "width: 1024px; height:400px; background: white;")


### Held-out predictions for example sites

Observed vs predicted log-density for the held-out years, with 90% prediction intervals.

In [ ]:
import (
	"encoding/csv"
	"fmt"
	"os"
	"sort"
	"strconv"

	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Load validation results and predictions
vf, _ := os.Open("../dat/validation_results.csv")
defer vf.Close()
vr := csv.NewReader(vf)
vr.Read()
valRecords, _ := vr.ReadAll()

pf, _ := os.Open("../dat/validation_predictions.csv")
defer pf.Close()
pr := csv.NewReader(pf)
pr.Read()
predRecords, _ := pr.ReadAll()

// Index predictions by site ID
type pred struct {
	year, obs, mean, lo90, hi90 float64
}
sitePreds := make(map[int][]pred)
for _, rec := range predRecords {
	id, _ := strconv.Atoi(rec[0])
	p := pred{}
	p.year, _ = strconv.ParseFloat(rec[1], 64)
	p.obs, _ = strconv.ParseFloat(rec[2], 64)
	p.mean, _ = strconv.ParseFloat(rec[3], 64)
	p.lo90, _ = strconv.ParseFloat(rec[4], 64)
	p.hi90, _ = strconv.ParseFloat(rec[5], 64)
	sitePreds[id] = append(sitePreds[id], p)
}

// Pick sites by RMSE: best, median, worst
type siteRMSE struct {
	id   int
	rmse float64
}
var sr []siteRMSE
for _, rec := range valRecords {
	if rec[8] != "OK" { continue }
	id, _ := strconv.Atoi(rec[0])
	rmse, _ := strconv.ParseFloat(rec[4], 64)
	sr = append(sr, siteRMSE{id, rmse})
}
sort.Slice(sr, func(i, j int) bool { return sr[i].rmse < sr[j].rmse })

// Load full site data for training period overlay
allSites := data.LoadAllSiteTimeSeries("../dat/brown_trout_panel_with_covariates.csv")

picks := []struct{ label string; idx int }{
	{"Best RMSE", 0},
	{"Q25 RMSE", len(sr) / 4},
	{"Median RMSE", len(sr) / 2},
	{"Q75 RMSE", 3 * len(sr) / 4},
	{"Worst RMSE", len(sr) - 1},
}

for _, pick := range picks {
	siteID := sr[pick.idx].id
	preds := sitePreds[siteID]
	sd := allSites[siteID]
	if len(preds) == 0 || sd == nil {
		continue
	}

	// Build dataframe: training observed + held-out observed + predicted mean
	var allYears, allValues []float64
	var allLabels []string

	// Training data
	nTrain := len(sd.Years) - len(preds)
	for t := 0; t < nTrain; t++ {
		allYears = append(allYears, sd.Years[t])
		allValues = append(allValues, sd.LogDensity[t][0])
		allLabels = append(allLabels, "Train")
	}

	// Held-out: observed, predicted, interval bounds
	for _, p := range preds {
		allYears = append(allYears, p.year, p.year, p.year, p.year)
		allValues = append(allValues, p.obs, p.mean, p.lo90, p.hi90)
		allLabels = append(allLabels, "Held-out obs", "Predicted", "90% lo", "90% hi")
	}

	df := dataframe.New(
		series.New(allYears, series.Float, "YEAR"),
		series.New(allValues, series.Float, "LOG_DENSITY"),
		series.New(allLabels, series.String, "TYPE"),
	)

	scatter := analysis.NewScatterPlotFromDataFrame(&df, "YEAR", "LOG_DENSITY", "TYPE")
	scatter.SetGlobalOptions(
		charts.WithTitleOpts(opts.Title{
			Title: fmt.Sprintf("%s — Site %d (RMSE=%.3f)", pick.label, siteID, sr[pick.idx].rmse),
			Bottom: "1%",
		}),
		charts.WithYAxisOpts(opts.YAxis{
			Min: floats.Min(allValues) - 0.5,
			Max: floats.Max(allValues) + 0.5,
		}),
		charts.WithXAxisOpts(opts.XAxis{
			Min: floats.Min(allYears) - 1,
			Max: floats.Max(allYears) + 1,
		}),
	)
	gonb_echarts.Display(scatter, "width: 1024px; height:350px; background: white;")
}
